# CW2 Code Dump Notebook (Appendix)

This notebook is a **code dump** for coursework appendix submission.

It prints all implemented project files with line numbers for PDF export.


In [47]:
from pathlib import Path
from IPython.display import display, Markdown
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
from IPython.display import HTML
HTML("""
<style>
body, .jp-RenderedHTMLCommon, .jp-Notebook, .jp-Cell, pre, code {
    font-size: 9px !important;
    line-height: 1.15 !important;
}
.jp-Cell {
    margin-top: 2px !important;
    margin-bottom: 2px !important;
}
pre {
    white-space: pre-wrap !important;
    word-break: break-word !important;
}
</style>
""")


REPO_ROOT = Path.cwd().parent
print('Working directory:', REPO_ROOT)


def print_file(path: str):
    p = REPO_ROOT / path
    display(Markdown(f"## `{path}`"))
    if not p.exists():
        print("MISSING")
        return
    # text = p.read_text(encoding="utf-8")
    print(p.read_text(encoding="utf-8"))
    # print(text)

Working directory: c:\Users\adirj\OneDrive\Documents\GitHub\Machine-Learning-KCL-CW2


## Source Code (`src/`)

In [48]:
print_file('src/config.py')

## `src/config.py`

from pathlib import Path
import yaml

def load_configurations(path):
    """Loads YAML configuration from disk into a Python dictionary."""
    with open(path,"r",encoding="utf-8") as f:
        return yaml.safe_load(f)


In [49]:
print_file('src/data.py')

## `src/data.py`

from pathlib import Path
from typing import Sequence
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


class SimCLRTransform:
    def __init__(self):
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(32, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply(
                [transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)],
                p=0.8,
            ),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.4914, 0.4822, 0.4465),
                std=(0.2023, 0.1994, 0.2010),
            ),
        ])

    def __call__(self, x):
        return self.transform(x), self.transform(x)


def get_classifier_train_transform():
    """Build train-time image augmentations for supervised classifier training."""
    return transforms.Compose([
        transforms.RandomCrop

In [50]:
print_file('src/clustering.py')

## `src/clustering.py`

import numpy as np
from sklearn.cluster import KMeans, MiniBatchKMeans

def cluster_embeddings(embeddings: np.ndarray,n_clusters:int, random_state:int) -> tuple[np.ndarray,np.ndarray]:
    """Clusters embeddings with KMeans/MiniBatchKMeans and returns labels and centroids."""
    if n_clusters <= 50:
        model = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    else:
        model = MiniBatchKMeans(n_clusters=n_clusters, random_state=random_state, n_init=10, batch_size=1024)

    return model.fit_predict(embeddings), model.cluster_centers_


In [51]:
print_file('src/typicality.py')

## `src/typicality.py`

import numpy as np
from sklearn.neighbors import NearestNeighbors

def compute_typicality_scores(embeddings: np.ndarray, k: int) -> np.ndarray:
    """Compute inverse average KNN distance as a typicality score."""
    k_eff = min(k + 1, len(embeddings))
    nn = NearestNeighbors(n_neighbors=k_eff, metric="euclidean")
    nn.fit(embeddings)
    distances, _ = nn.kneighbors(embeddings)

    neighbor_distances = distances[:, 1:] if k_eff > 1 else distances
    avg_dist = neighbor_distances.mean(axis=1)
    return 1.0 / (avg_dist + 1e-12)


def compute_cluster_aware_scores(cluster_embeddings: np.ndarray,centroid: np.ndarray,k: int,alpha: float) -> np.ndarray:
    """Blend typicality with centroid proximity for cluster-aware ranking"""
    typ = compute_typicality_scores(cluster_embeddings, k)

    dists = np.linalg.norm(cluster_embeddings - centroid[None, :], axis=1)
    centrality = 1.0 / (dists + 1e-12)

    def normalize(x: np.ndarray) -> np.ndarray:
        if np.allclose(x.max(), x.mi

In [52]:
print_file('src/selectors.py')

## `src/selectors.py`

import numpy as np

from .typicality import compute_typicality_scores, compute_cluster_aware_scores


def random_selector(num_samples: int, budget: int, rng: np.random.Generator) -> np.ndarray:
    """Randomly sample unique pool indices for the query batch."""
    selected = rng.choice(num_samples, size=budget, replace=False)
    assert len(np.unique(selected)) == len(selected)
    return np.sort(selected)
    


def tpcrand_selector(cluster_labels: np.ndarray, budget: int, rng: np.random.Generator) -> np.ndarray:
    """Select one random sample per cluster for TPCRand ablation."""
    selected = []
    for cluster_id in range(budget):
        members = np.where(cluster_labels == cluster_id)[0]
        if len(members) == 0:
            continue
        selected.append(rng.choice(members))

    assert len(np.unique(selected)) == len(selected)
    return np.array(sorted(selected), dtype=int)


def tpcrp_selector(embeddings: np.ndarray,cluster_labels: np.ndarray,budget: int,knn_k: int) ->

In [53]:
print_file('src/models.py')

## `src/models.py`

import torch
import torch.nn as nn
from torchvision.models import resnet18


def build_cifar_resnet18(num_classes: int | None = None) -> nn.Module:
    """
    CIFAR-style ResNet-18 stem:
    - 3x3 conv, stride 1
    - no initial maxpool
    """
    model = resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    if num_classes is not None:
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = build_cifar_resnet18()
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.output_dim = in_features

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


class ProjectionHead(nn.Module):
    def __init__(self, in_dim, proj_dim = 128):
        super().__init__()
        

In [54]:
print_file('src/simclr.py')

## `src/simclr.py`

import torch
import torch.nn.functional as F
from torch import nn
from tqdm import tqdm


class NTXentLoss(nn.Module):
    def __init__(self, temperature: float = 0.5) -> None:
        super().__init__()
        self.temperature = temperature

    def forward(self, z1: torch.Tensor, z2: torch.Tensor) -> torch.Tensor:
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)

        z = torch.cat([z1, z2], dim=0)
        sim = torch.matmul(z, z.T) / self.temperature

        batch_size = z1.size(0)
        mask = torch.eye(2 * batch_size, device=z.device, dtype=torch.bool)
        sim = sim.masked_fill(mask, float("-inf"))

        positives = torch.cat([
            torch.diag(sim, batch_size),
            torch.diag(sim, -batch_size),
        ])

        denominator = torch.logsumexp(sim, dim=1)
        loss = -positives + denominator
        return loss.mean()


def train_simclr_epoch(model, loader, optimizer, criterion, device):
    """Run one SimCLR training epoch an

In [55]:
print_file('src/train_classifier.py')

## `src/train_classifier.py`

from pathlib import Path
from typing import Any

import torch
import torch.nn as nn
from torch.optim import SGD
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

from .models import CIFARClassifier

def train_one_epoch(model:nn.Module, loader: torch.utils.data.DataLoader,optimiser: torch.optim.Optimizer,criterion: nn.Module,device: torch.device):
    """Train classifier for one epoch and return loss/accuracy metrics"""
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images,targets in tqdm(loader, leave=False,desc="Train"):
        images = images.to(device,non_blocking=True)
        targets = targets.to(device,non_blocking=True)

        optimiser.zero_grad()
        logits = model(images)
        loss = criterion(logits,targets)
        loss.backward()
        optimiser.step()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == targets).sum()

In [56]:
print_file('src/embeddings.py')

## `src/embeddings.py`

import numpy as np
import torch
import torch.nn.functional
from torch.utils.data import DataLoader
from tqdm import tqdm

@torch.no_grad()
def grab_embeddings(encoder,loader:DataLoader,device:torch.device) -> np.ndarray:
    """Extract and stack encoder embeddings for all batches in a loader"""
    encoder.eval()
    chunks = []

    for x,y in tqdm(loader,leave=False):
        x = x.to(device)
        feats = encoder(x)
        feats = torch.nn.functional.normalize(feats,p=2,dim=1)
        chunks.append(feats.cpu().numpy())

    return np.concatenate(chunks,axis=0)


In [57]:
print_file('src/evaluate.py')

## `src/evaluate.py`

import numpy as np

def compute_class_distribution(labels,num_classes=10):
    """Compute per class counts and proportions for a label array."""
    counts = np.bincount(labels,minlength=num_classes)
    return counts / counts.sum()

def summarise_labels(labels,num_classes=10):
    """Return a compact class distribution summary for selected labels"""
    counts = np.bincount(labels,minlength=num_classes)
    proportions = counts / counts.sum()
    return {"counts":counts.tolist(), "proportions":proportions.tolist()} 


In [58]:
print_file('src/seed.py')

## `src/seed.py`

import os
import random
import numpy as np
import torch

def set_seed(seed):
    """Sets relevant seeds for reproducible experiment behaviour."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [59]:
print_file('src/experiment.py')

## `src/experiment.py`

import csv
import json
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, log_loss
from sklearn.semi_supervised import LabelSpreading
from torch.optim import SGD
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader

from .clustering import cluster_embeddings
from .config import load_configurations
from .data import (
    get_cifar10_test,
    get_cifar10_train,
    get_classifier_train_transform,
    get_eval_transform,
    make_subset_loader,
)
from .embeddings import grab_embeddings
from .evaluate import summarise_labels
from .models import SimCLRModel
from .seed import set_seed
from .selectors import (
    kcenter_selector,
    random_selector,
    tpcinv_selector,
    tpcnoclust_selector,
    tpcrp_ccfl_selector,
    tpcrand_selector,
    tpcrp_modified_selector,
    tpcrp_selector,
)
from .typicality import comput

## Scripts (`scripts/`)

In [60]:
print_file('scripts/train_simclr.py')

## `scripts/train_simclr.py`

from pathlib import Path

import torch
from torch.optim import SGD
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader

from src.config import load_configurations
from src.data import SimCLRTransform, get_cifar10_train
from src.models import SimCLRModel
from src.seed import set_seed
from src.simclr import NTXentLoss, train_simclr_epoch


def main() -> None:
    """Train SimCLR encoder on CIFAR-10 and save checkpoint"""
    cfg = load_configurations("configs/default.yaml")
    set_seed(cfg["seed"])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:",device)
    simclr_cfg = cfg["simclr"]
    data_cfg = cfg["data"]

    dataset = get_cifar10_train(
        root=data_cfg["root"],
        transform=SimCLRTransform(),
    )
    loader = DataLoader(dataset,batch_size=simclr_cfg["batch_size"],shuffle=True,num_workers=data_cfg["num_workers"],pin_memory=True,drop_last=True)

    model = SimCLRModel(proj_

In [61]:
print_file('scripts/run_experiments.py')

## `scripts/run_experiments.py`

from src.config import load_configurations
from src.experiment import run_single_experiment


def main() -> None:
    """Run configured experiment grid across frameworks, methods, budgets and seeds."""
    cfg = load_configurations("configs/default.yaml")

    methods = cfg["experiment"]["methods"]
    budgets = cfg["selection"]["budgets"]
    seeds = cfg["experiment"]["seeds"]
    frameworks = cfg["experiment"].get("frameworks", [cfg["experiment"].get("framework", "fully_supervised")])

    for framework in frameworks:
        for seed in seeds:
            for budget in budgets:
                for method in methods:
                    print("\n" + "=" * 80)
                    print(
                        f"Running framework={framework}, method={method}, "
                        f"budget={budget}, seed={seed}"
                    )
                    print("=" * 80)
                    run_single_experiment(
                        config_path="configs/default.yaml",
          

In [62]:
print_file('scripts/aggregate_results.py')

## `scripts/aggregate_results.py`

from pathlib import Path
import pandas as pd
from pandas.errors import EmptyDataError

MAIN_METHODS = ["random", "tpcrand", "tpcrp", "tpcinv", "tpcnoclust", "tpcrp_ccfl"]
MOD_METHODS = ["tpcrp", "tpcrp_ccfl"]

def format_mean_std(mean: float, std: float) -> str:
    """Format mean and std accuracy as a percent plus-minus string."""
    return f"{mean * 100:.2f} $\\pm$ {std * 100:.2f}"


def dedupe(df: pd.DataFrame) -> pd.DataFrame:
    dedup_keys = ["method", "budget", "seed"]
    if "framework" in df.columns:
        dedup_keys = ["framework"] + dedup_keys
    return df.drop_duplicates(subset=dedup_keys, keep="last")


def _aggregate_metrics(df: pd.DataFrame) -> pd.DataFrame:
    group_keys = ["method", "budget"]
    if "framework" in df.columns:
        group_keys = ["framework"] + group_keys

    grouped = (
        df.groupby(group_keys, as_index=False)
        .agg(
            best_mean=("best_test_accuracy", "mean"),
            best_std=("best_test_accuracy", "std"),
          

In [63]:
print_file('scripts/run_stats.py')

## `scripts/run_stats.py`

from __future__ import annotations

from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from pandas.errors import EmptyDataError
from scipy.stats import ttest_rel, wilcoxon

COMPARISONS = [
    ("tpcrp", "random"),
    ("tpcrp", "tpcrand"),
    ("tpcrp_ccfl", "tpcrp"),
]


def dedupe(df: pd.DataFrame) -> pd.DataFrame:
    dedup_keys = ["method", "budget", "seed"]
    if "framework" in df.columns:
        dedup_keys = ["framework"] + dedup_keys
    return df.drop_duplicates(subset=dedup_keys, keep="last")


def cohens_dz(diff: np.ndarray) -> float:
    """Compute Cohen's dz for paired samples."""
    if diff.size < 2:
        return float("nan")
    sd = float(np.std(diff, ddof=1))
    if sd <= 1e-12:
        # All differences (almost) identical: dz undefined/infinite in theory.
        return 0.0 if abs(float(np.mean(diff))) <= 1e-12 else float("nan")
    return float(np.mean(diff) / sd)


def paired_row(sub: pd.DataFrame, method_a: str, method_b: st

In [64]:
print_file('scripts/make_plots.py')

## `scripts/make_plots.py`

from __future__ import annotations

from pathlib import Path
from typing import Sequence

import matplotlib
import pandas as pd
from pandas.errors import EmptyDataError

matplotlib.use("Agg")
import matplotlib.pyplot as plt

ABLATION_METHODS = ["random", "tpcrand", "tpcrp", "tpcinv", "tpcnoclust"]
MODIFICATION_METHODS = ["tpcrp", "tpcrp_ccfl"]
SECONDARY_METHODS = ["kcenter", "uncertainty", "margin", "entropy", "dbal", "bald", "badge"]

def build_global_df_from_raw_metrics(metrics_path: Path) -> pd.DataFrame:
    """Recompute pooled method/budget means and SEs from raw metrics, not from already-aggregated framework rows."""
    if not metrics_path.exists():
        raise FileNotFoundError(f"Could not find raw metrics file at {metrics_path}")

    raw_df = pd.read_csv(metrics_path)

    required_cols = {"method", "budget", "seed", "best_test_accuracy"}
    missing = required_cols - set(raw_df.columns)
    if missing:
        raise ValueError(f"Raw metrics CSV is missing required columns:

## Configuration

In [65]:
print_file('configs/default.yaml')

## `configs/default.yaml`

seed: 21

data:
  name: "cifar10"
  root: "./data"
  num_workers: 2

simclr:
  batch_size: 256
  epochs: 100
  lr: 0.4
  momentum: 0.9
  nesterov: false
  min_lr: 0.000001
  weight_decay: 0.0001
  temperature: 0.5
  projection_dim: 128
  embedding_dim: 512
  save_path: "./results/checkpoints/simclr_resnet18.pt"

selection:
  knn_k: 20
  budgets: [10, 20, 50, 100, 200]
  clustering: "kmeans"
  modified_alpha: 0.7

classifier:
  backbone: "resnet18"
  batch_size: 64
  epochs: 30
  lr: 0.025
  momentum: 0.9
  weight_decay: 0.0005

experiment:
  frameworks: ["fully_supervised", "ssl_embedding", "semi_supervised"]
  methods: ["kcenter", "uncertainty", "margin","entropy", "dbal","bald", "badge","random", "tpcrand", "tpcrp", "tpcinv", "tpcnoclust", "tpcrp_ccfl"]
  seeds: [42,43,44]
  iterative_rounds: 2
  initial_labeled: 0
  max_clusters: 500
  min_cluster_size: 5
  mc_passes: 10
  mc_dropout_p: 0.2

